# PostgreSQL — Complete Detailed Notes

> PostgreSQL is an open-source object-relational database management system (ORDBMS) known for strong SQL support, ACID transactions, MVCC, extensibility, advanced data types, indexing, JSONB, full-text search, and extensions such as PostGIS and pgvector.

---

## 1. What Is PostgreSQL?

PostgreSQL is a production-grade relational database system used to store, query, protect, and process structured data.

It is commonly used for:

- Backend applications
- E-commerce
- Banking and financial systems
- SaaS platforms
- Analytics
- APIs
- Data-intensive applications
- RAG systems
- Agentic AI systems

### PostgreSQL Mental Model

```text
Application
    │
    ▼
Database Driver / ORM
    │
    ▼
PostgreSQL Server
    │
    ├── Parser
    ├── Planner / Optimizer
    ├── Executor
    ├── Buffer / Cache
    ├── Transaction Manager
    ├── WAL
    └── Storage
```

---

# 2. SQL vs PostgreSQL

SQL is a language.

PostgreSQL is a database management system that implements SQL and adds many advanced capabilities.

```text
SQL
 │
 └── Language

PostgreSQL
 │
 ├── SQL Engine
 ├── Transaction System
 ├── Storage
 ├── Indexes
 ├── JSONB
 ├── Full-Text Search
 ├── Extensions
 └── Vector Support
```

---

# 3. PostgreSQL Architecture

A simplified architecture is:

```text
┌───────────────────────────────┐
│          Application          │
│ Python / Java / Node / API    │
└──────────────┬────────────────┘
               │
               ▼
┌───────────────────────────────┐
│       PostgreSQL Server       │
│                               │
│  Connection / Session         │
│          ↓                    │
│       SQL Parser              │
│          ↓                    │
│   Planner / Optimizer         │
│          ↓                    │
│       Executor                │
│          ↓                    │
│ Buffer / Cache / Storage      │
└──────────────┬────────────────┘
               │
               ▼
┌───────────────────────────────┐
│ Persistent Data + WAL         │
└───────────────────────────────┘
```

The general query lifecycle is:

```text
SQL
 ↓
Parse
 ↓
Analyze
 ↓
Plan
 ↓
Execute
 ↓
Return Result
```

---

# 4. Database, Schema, Table

PostgreSQL can be viewed as:

```text
PostgreSQL Server
      │
      ▼
   Database
      │
      ▼
    Schema
      │
      ├── Tables
      ├── Views
      ├── Functions
      ├── Sequences
      └── Types
```

Example:

```text
company_db
 ├── app
 │    ├── users
 │    ├── orders
 │    └── products
 │
 ├── analytics
 │    ├── sales_daily
 │    └── customer_metrics
 │
 └── audit
      └── audit_logs
```

A schema is a namespace. `app.users` and `audit.users` can exist as separate objects.

---

# 5. Tables, Rows, and Columns

A table stores records.

```sql
CREATE TABLE users (
    user_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    created_at TIMESTAMPTZ DEFAULT NOW()
);
```

Conceptually:

```text
users
┌─────────┬──────────┬────────────────────┬────────────────────┐
│ user_id │ name     │ email              │ created_at         │
├─────────┼──────────┼────────────────────┼────────────────────┤
│ 1       │ Aisha    │ aisha@example.com  │ 2026-09-01         │
│ 2       │ Rahul    │ rahul@example.com  │ 2026-09-02         │
└─────────┴──────────┴────────────────────┴────────────────────┘
```

A row represents one entity instance.

A column represents one attribute.

---

# 6. PostgreSQL Data Types

Important types include:

```text
INTEGER / BIGINT
NUMERIC / DECIMAL
REAL / DOUBLE PRECISION
TEXT / VARCHAR
BOOLEAN
DATE
TIME
TIMESTAMP
TIMESTAMPTZ
INTERVAL
UUID
JSON
JSONB
ARRAY
ENUM
```

### Numeric

```sql
quantity INTEGER;
user_id BIGINT;
price NUMERIC(12,2);
```

Use `NUMERIC` for exact decimal calculations such as financial values.

### Text

```sql
name TEXT;
description TEXT;
```

PostgreSQL `TEXT` is often practical when a strict maximum length is not required.

### Boolean

```sql
is_active BOOLEAN DEFAULT TRUE;
```

### Timestamp

```sql
created_at TIMESTAMPTZ DEFAULT NOW();
```

For applications spanning time zones, `TIMESTAMPTZ` is often preferable for real-world instants.

---

# 7. UUID

UUIDs provide globally unique identifiers.

```sql
CREATE EXTENSION IF NOT EXISTS pgcrypto;

CREATE TABLE users (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    name TEXT NOT NULL
);
```

Example:

```text
550e8400-e29b-41d4-a716-446655440000
```

UUIDs can be useful when sequential numeric IDs should not be exposed externally.

---

# 8. JSON and JSONB

PostgreSQL supports both `JSON` and `JSONB`.

`JSON` stores JSON text.

`JSONB` stores a binary representation optimized for processing and indexing.

Example:

```sql
CREATE TABLE products (
    product_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    metadata JSONB
);
```

Insert:

```sql
INSERT INTO products (name, metadata)
VALUES (
    'Laptop',
    '{"brand": "Dell", "ram": 16, "storage": 512}'
);
```

Query:

```sql
SELECT metadata->>'brand'
FROM products;
```

Result:

```text
Dell
```

For many application workloads, JSONB is the more useful choice.

---

# 9. JSONB Operators

Important operators:

```text
->      JSON object/array
->>     JSON value as text
@>      contains
?       key exists
```

Example:

```sql
SELECT metadata->>'brand'
FROM products;
```

Nested:

```sql
SELECT metadata->'specs'->>'ram'
FROM products;
```

Containment:

```sql
SELECT *
FROM products
WHERE metadata @> '{"brand": "Dell"}';
```

---

# 10. Arrays

PostgreSQL supports arrays.

```sql
CREATE TABLE users (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT,
    skills TEXT[]
);
```

Insert:

```sql
INSERT INTO users (name, skills)
VALUES ('Aisha', ARRAY['Python', 'SQL', 'PostgreSQL']);
```

Query:

```sql
SELECT *
FROM users
WHERE 'Python' = ANY(skills);
```

Arrays are useful for simple embedded lists. If elements need independent relationships, a normalized child table is often better.

---

# 11. ENUM

```sql
CREATE TYPE order_status AS ENUM (
    'pending',
    'paid',
    'shipped',
    'delivered',
    'cancelled'
);
```

Use:

```sql
CREATE TABLE orders (
    order_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    status order_status NOT NULL
);
```

ENUM is useful for a controlled finite state set.

---

# 12. Primary Key

A primary key uniquely identifies a row.

```sql
CREATE TABLE users (
    user_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL
);
```

A primary key is:

- Unique
- Not null
- Used to identify records
- Backed by uniqueness enforcement

---

# 13. Foreign Key

Foreign keys establish relationships.

```sql
CREATE TABLE customers (
    customer_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE orders (
    order_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id BIGINT NOT NULL
        REFERENCES customers(customer_id)
);
```

Relationship:

```text
customers
    │
    │ 1
    ▼
orders
    │
    └── customer_id
```

Foreign keys enforce referential integrity.

---

# 14. One-to-One

```text
User
 │
 │ 1
 ▼
Profile
```

Example:

```sql
CREATE TABLE user_profiles (
    user_id BIGINT PRIMARY KEY REFERENCES users(user_id),
    bio TEXT
);
```

---

# 15. One-to-Many

Example:

```text
Customer
   │
   ├── Order 1
   ├── Order 2
   └── Order 3
```

The many-side stores the foreign key:

```sql
orders.customer_id
```

---

# 16. Many-to-Many

Use a junction table.

```text
Students
   │
   ▼
student_courses
   ▲
   │
Courses
```

Example:

```sql
CREATE TABLE student_courses (
    student_id BIGINT REFERENCES students(id),
    course_id BIGINT REFERENCES courses(id),
    PRIMARY KEY (student_id, course_id)
);
```

---

# 17. Constraints

Important constraints:

```text
PRIMARY KEY
FOREIGN KEY
UNIQUE
NOT NULL
CHECK
DEFAULT
```

Example:

```sql
CREATE TABLE employees (
    employee_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    salary NUMERIC(12,2) CHECK (salary >= 0),
    is_active BOOLEAN DEFAULT TRUE
);
```

Constraints are database-level protection against invalid data.

---

# 18. CRUD

CRUD means:

```text
C → Create
R → Read
U → Update
D → Delete
```

## INSERT

```sql
INSERT INTO users (name, email)
VALUES ('Aisha', 'aisha@example.com');
```

## SELECT

```sql
SELECT name, email
FROM users;
```

## UPDATE

```sql
UPDATE users
SET name = 'Aisha Khan'
WHERE user_id = 1;
```

## DELETE

```sql
DELETE FROM users
WHERE user_id = 1;
```

Always be careful with `UPDATE` and `DELETE` without `WHERE`.

---

# 19. RETURNING

PostgreSQL supports `RETURNING`.

```sql
INSERT INTO users (name, email)
VALUES ('Aisha', 'aisha@example.com')
RETURNING user_id, created_at;
```

It is extremely useful in backend applications because generated values can be returned immediately.

---

# 20. UPSERT

PostgreSQL supports:

```sql
INSERT ... ON CONFLICT
```

Example:

```sql
INSERT INTO users (email, name)
VALUES ('aisha@example.com', 'Aisha')
ON CONFLICT (email)
DO UPDATE
SET name = EXCLUDED.name;
```

Meaning:

```text
New record?
   │
   ├── Yes → INSERT
   │
   └── Conflict → UPDATE
```

Useful for synchronization and idempotent writes.

---

# 21. Filtering

```sql
SELECT *
FROM products
WHERE price > 50000;
```

Operators:

```text
=
!=
<>
>
<
>=
<=
```

Useful operators:

```sql
WHERE city IN ('Mumbai', 'Pune');

WHERE price BETWEEN 50000 AND 100000;

WHERE name ILIKE '%laptop%';
```

---

# 22. NULL

`NULL` means missing/unknown.

Incorrect:

```sql
WHERE email = NULL;
```

Correct:

```sql
WHERE email IS NULL;
```

or:

```sql
WHERE email IS NOT NULL;
```

---

# 23. COALESCE

`COALESCE` returns the first non-null expression.

```sql
SELECT
    name,
    COALESCE(phone, 'Not provided') AS phone
FROM users;
```

---

# 24. ORDER BY, LIMIT, OFFSET

```sql
SELECT *
FROM products
ORDER BY price DESC
LIMIT 10;
```

Pagination:

```sql
SELECT *
FROM products
ORDER BY product_id
LIMIT 20
OFFSET 40;
```

Large offsets can become inefficient. Keyset pagination is often better at scale.

---

# 25. Aggregation

Important functions:

```text
COUNT
SUM
AVG
MIN
MAX
```

Example:

```sql
SELECT COUNT(*)
FROM orders;
```

---

# 26. GROUP BY and HAVING

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY customer_id;
```

Filter groups:

```sql
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 5;
```

Remember:

```text
WHERE  → filters rows
HAVING → filters groups
```

---

# 27. Joins

Important joins:

```text
INNER JOIN
LEFT JOIN
RIGHT JOIN
FULL OUTER JOIN
CROSS JOIN
SELF JOIN
```

## INNER JOIN

```sql
SELECT
    o.order_id,
    c.name
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id;
```

Only matching records are returned.

## LEFT JOIN

```sql
SELECT
    c.customer_id,
    c.name,
    o.order_id
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id;
```

Useful for finding all customers, including those with no orders.

---

# 28. SELF JOIN

Hierarchical data can use a self join.

```sql
SELECT
    e.name AS employee,
    m.name AS manager
FROM employees e
LEFT JOIN employees m
    ON e.manager_id = m.employee_id;
```

Diagram:

```text
Manager
   │
   ├── Employee A
   ├── Employee B
   └── Employee C
```

---

# 29. Subqueries

Example:

```sql
SELECT *
FROM products
WHERE price > (
    SELECT AVG(price)
    FROM products
);
```

This returns products priced above average.

---

# 30. EXISTS

```sql
SELECT *
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.customer_id = c.customer_id
);
```

This finds customers who have at least one order.

---

# 31. CTE

CTE means Common Table Expression.

```sql
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_orders
WHERE order_count > 5;
```

CTEs improve readability and can help structure complex queries.

---

# 32. Recursive CTE

Useful for hierarchical data:

- Organization charts
- Categories
- Folder structures
- Parent-child relationships

Conceptual pattern:

```sql
WITH RECURSIVE hierarchy AS (
    -- base rows

    UNION ALL

    -- recursive rows
)
SELECT *
FROM hierarchy;
```

---

# 33. Window Functions

Window functions calculate across related rows without collapsing them.

Important functions:

```text
ROW_NUMBER
RANK
DENSE_RANK
LAG
LEAD
SUM() OVER
AVG() OVER
```

Example:

```sql
SELECT
    employee_id,
    department,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS row_num
FROM employees;
```

---

# 34. RANK vs DENSE_RANK

For values:

```text
100
100
90
```

`RANK()`:

```text
1
1
3
```

`DENSE_RANK()`:

```text
1
1
2
```

---

# 35. LAG and LEAD

```sql
SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_revenue
FROM monthly_sales;
```

Useful for:

- Month-over-month growth
- Time-series analysis
- Trend comparison

---

# 36. Transactions

A transaction groups operations into one logical unit.

```sql
BEGIN;

UPDATE accounts
SET balance = balance - 1000
WHERE account_id = 1;

UPDATE accounts
SET balance = balance + 1000
WHERE account_id = 2;

COMMIT;
```

On failure:

```sql
ROLLBACK;
```

---

# 37. ACID

## Atomicity

All operations succeed or none are committed.

## Consistency

Database constraints and rules remain valid.

## Isolation

Concurrent transactions are isolated according to the chosen isolation level.

## Durability

Committed changes are designed to survive failures through PostgreSQL's durability mechanisms.

---

# 38. Transaction Isolation

PostgreSQL supports:

```text
READ COMMITTED
REPEATABLE READ
SERIALIZABLE
```

Isolation controls how concurrent transactions observe each other's work.

Higher isolation can provide stronger guarantees but may require handling more contention or transaction retries.

---

# 39. MVCC

MVCC means **Multi-Version Concurrency Control**.

PostgreSQL maintains row versions and visibility information so concurrent readers and writers can work without unnecessary blocking.

Conceptually:

```text
Transaction A
    │
    └── sees version 1

Transaction B
    │
    └── creates version 2

Visibility rules
    │
    └── determine what each transaction sees
```

MVCC is a foundational PostgreSQL concurrency feature.

---

# 40. VACUUM and Autovacuum

Because MVCC creates old row versions, PostgreSQL needs maintenance.

`VACUUM` helps clean up obsolete row versions and maintain storage health.

`autovacuum` automates routine maintenance.

```text
UPDATE / DELETE
      ↓
Old row version
      ↓
No longer needed
      ↓
VACUUM
      ↓
Space becomes reusable
```

---

# 41. ANALYZE

`ANALYZE` collects statistics used by the query planner.

```sql
ANALYZE users;
```

Statistics help PostgreSQL estimate:

- Row counts
- Data distribution
- Selectivity
- Query costs

---

# 42. Indexes

An index helps PostgreSQL find rows efficiently.

Without an index:

```text
Query
 ↓
Scan many rows
 ↓
Find target
```

With an index:

```text
Query
 ↓
Index
 ↓
Candidate rows
 ↓
Table data
```

Indexes improve reads but add storage and write/maintenance overhead.

---

# 43. B-Tree Index

B-tree is the default and most common index type.

Useful for:

```text
=
<
>
<=
>=
ORDER BY
range conditions
```

Example:

```sql
CREATE INDEX idx_users_email
ON users(email);
```

---

# 44. Composite Index

```sql
CREATE INDEX idx_orders_customer_date
ON orders(customer_id, created_at);
```

This can help queries such as:

```sql
SELECT *
FROM orders
WHERE customer_id = 101
ORDER BY created_at DESC;
```

Column order matters.

---

# 45. Partial Index

```sql
CREATE INDEX idx_active_users_email
ON users(email)
WHERE is_active = TRUE;
```

Only active users are indexed.

Useful when a query frequently targets a subset of a table.

---

# 46. Expression Index

```sql
CREATE INDEX idx_users_lower_email
ON users(LOWER(email));
```

Then:

```sql
SELECT *
FROM users
WHERE LOWER(email) = 'aisha@example.com';
```

The expression index can support this access pattern.

---

# 47. GIN Index

GIN is useful for:

- JSONB
- Arrays
- Full-text search

Example:

```sql
CREATE INDEX idx_products_metadata
ON products
USING GIN(metadata);
```

Then:

```sql
SELECT *
FROM products
WHERE metadata @> '{"brand": "Dell"}';
```

---

# 48. GiST and BRIN

**GiST** is a flexible index framework used by various data types and extensions, including many PostGIS use cases.

**BRIN** is a compact index useful for very large tables where physical row order correlates with the indexed value.

Example:

```text
events
 ├── created_at
 ├── event_type
 └── payload

BRIN on created_at
      ↓
Very small index
      ↓
Efficient filtering when data is naturally ordered
```

---

# 49. EXPLAIN

Use `EXPLAIN` to inspect the query plan.

```sql
EXPLAIN
SELECT *
FROM users
WHERE email = 'aisha@example.com';
```

Possible plan operations include:

```text
Seq Scan
Index Scan
Bitmap Index Scan
Bitmap Heap Scan
Nested Loop
Hash Join
Merge Join
```

---

# 50. EXPLAIN ANALYZE

```sql
EXPLAIN ANALYZE
SELECT *
FROM orders
WHERE customer_id = 101;
```

`EXPLAIN` estimates a plan.

`EXPLAIN ANALYZE` executes the query and reports actual runtime statistics.

Use it carefully with data-modifying statements because those statements execute.

---

# 51. Query Planner

The planner chooses an execution strategy.

```text
SQL
 ↓
Parser
 ↓
Query Representation
 ↓
Planner / Optimizer
 ↓
Execution Plan
 ↓
Executor
 ↓
Result
```

It considers statistics, indexes, estimated row counts, join strategies, sorting, and costs.

---

# 52. Sequential Scan Is Not Always Bad

For a tiny table, reading the whole table can be cheaper than using an index.

Therefore:

> A sequential scan is not automatically a performance problem.

Performance should be measured using the actual workload.

---

# 53. Join Algorithms

PostgreSQL can use:

```text
Nested Loop
Hash Join
Merge Join
```

### Nested Loop

Good when one side is small and the other side has an appropriate index.

### Hash Join

Useful for many equality joins.

### Merge Join

Useful when inputs are sorted or sorting is efficient.

The planner chooses based on estimated cost.

---

# 54. Views

A view is a stored query.

```sql
CREATE VIEW active_customers AS
SELECT
    customer_id,
    name,
    email
FROM customers
WHERE is_active = TRUE;
```

Then:

```sql
SELECT *
FROM active_customers;
```

Views provide abstraction and reusable query logic.

---

# 55. Materialized Views

A materialized view stores query results physically.

```sql
CREATE MATERIALIZED VIEW monthly_sales AS
SELECT
    DATE_TRUNC('month', created_at) AS month,
    SUM(amount) AS revenue
FROM orders
GROUP BY 1;
```

Refresh:

```sql
REFRESH MATERIALIZED VIEW monthly_sales;
```

Useful for expensive reports that do not require real-time results.

---

# 56. Functions

PostgreSQL supports functions.

```sql
CREATE FUNCTION add_numbers(a INTEGER, b INTEGER)
RETURNS INTEGER
LANGUAGE SQL
AS $$
    SELECT a + b;
$$;
```

Call:

```sql
SELECT add_numbers(10, 20);
```

Functions encapsulate reusable database logic.

---

# 57. PL/pgSQL

PL/pgSQL provides procedural programming.

```sql
CREATE OR REPLACE FUNCTION calculate_discount(price NUMERIC)
RETURNS NUMERIC
LANGUAGE plpgsql
AS $$
BEGIN
    IF price >= 10000 THEN
        RETURN price * 0.90;
    ELSE
        RETURN price;
    END IF;
END;
$$;
```

It supports variables, conditions, loops, SQL, and exception handling.

---

# 58. Procedures

Procedures are invoked using:

```sql
CALL procedure_name(...);
```

They are useful for operational database-side workflows.

Functions and procedures have different semantics and invocation patterns.

---

# 59. Triggers

A trigger automatically invokes database logic when an event occurs.

```text
INSERT / UPDATE / DELETE
          ↓
        Trigger
          ↓
    Trigger Function
          ↓
        Action
```

Common uses:

- Audit logging
- Derived values
- Change tracking
- Enforcement of business rules

---

# 60. Full-Text Search

PostgreSQL supports full-text search using:

```text
tsvector
tsquery
```

Example:

```sql
SELECT *
FROM articles
WHERE to_tsvector('english', content)
      @@ plainto_tsquery('english', 'machine learning');
```

This is generally more appropriate than large-scale `%LIKE%` searches for natural-language document search.

---

# 61. Full-Text Search Index

```sql
CREATE INDEX idx_articles_search
ON articles
USING GIN (
    to_tsvector('english', content)
);
```

This can accelerate full-text search.

---

# 62. Extensions

PostgreSQL is highly extensible.

Common extensions include:

```text
pgvector
PostGIS
pg_trgm
citext
uuid-ossp
```

Extensions can add:

- Data types
- Operators
- Functions
- Indexing
- Specialized capabilities

---

# 63. pg_trgm

`pg_trgm` supports trigram-based similarity.

Useful for:

- Fuzzy search
- Typo-tolerant matching
- Similarity
- Some `LIKE` / `ILIKE` workloads

Conceptually:

```text
"postgres"
    ≈
"postgre"
```

---

# 64. PostGIS

PostGIS adds geospatial capabilities.

Use cases:

- Maps
- Delivery
- Ride-sharing
- Nearby stores
- Location search
- Logistics

```text
Application
    ↓
PostgreSQL
    ↓
PostGIS
    ↓
Spatial Data + Spatial Indexes
```

---

# 65. pgvector

`pgvector` adds vector storage and similarity search.

This is highly relevant to RAG and Agentic AI.

```text
Documents
   ↓
Chunking
   ↓
Embedding Model
   ↓
Vectors
   ↓
PostgreSQL + pgvector
   ↓
Similarity Search
   ↓
Relevant Chunks
   ↓
LLM
```

Example schema:

```sql
CREATE TABLE document_chunks (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    document_id BIGINT NOT NULL,
    content TEXT NOT NULL,
    metadata JSONB,
    embedding VECTOR(1536)
);
```

The vector dimension must match the embedding model.

---

# 66. Vector Retrieval

Conceptually:

```sql
SELECT
    id,
    content,
    metadata
FROM document_chunks
ORDER BY embedding <=> :query_embedding
LIMIT 5;
```

The exact operator and index depend on the selected similarity metric and pgvector configuration.

---

# 67. PostgreSQL + RAG Architecture

```text
Documents
    ↓
Document Loader
    ↓
Text Splitter
    ↓
Embedding Model
    ↓
┌─────────────────────────┐
│ PostgreSQL + pgvector   │
│                         │
│ chunks                  │
│ embeddings              │
│ metadata                │
└────────────┬────────────┘
             │
             ▼
      Similarity Search
             │
             ▼
       Relevant Chunks
             │
             ▼
            LLM
             │
             ▼
        Final Answer
```

---

# 68. PostgreSQL + LangGraph

A LangGraph agent can use PostgreSQL for:

- Users
- Conversations
- Messages
- Documents
- Tool logs
- Audit records
- Application state
- Embeddings

Architecture:

```text
User
 ↓
FastAPI
 ↓
LangGraph
 ├── LLM
 ├── Tools
 ├── Retriever
 └── State
       ↓
   PostgreSQL
   ├── Relational Data
   ├── JSONB
   ├── Audit Logs
   └── pgvector
```

A crucial principle is to expose controlled database capabilities through application services/tools rather than giving an LLM unrestricted SQL access.

---

# 69. PostgreSQL + FastAPI

Common backend architecture:

```text
Client
  ↓
FastAPI
  ↓
Service Layer
  ↓
Repository / Data Access
  ↓
SQLAlchemy
  ↓
Driver
  ↓
PostgreSQL
```

This separates:

- API logic
- Business logic
- Persistence logic

---

# 70. PostgreSQL + SQLAlchemy

SQLAlchemy is a Python SQL toolkit and ORM.

```text
Python Application
       ↓
   SQLAlchemy
       ↓
   psycopg / asyncpg
       ↓
   PostgreSQL
```

A model can look like:

```python
from sqlalchemy import String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
```

A strong backend developer should understand both ORM behavior and SQL.

---

# 71. Connection Pooling

Opening a new database connection for every request is expensive.

```text
Application
    ↓
Connection Pool
 ┌──┼──┬──┐
 C1 C2 C3 C4
 └──┼──┴──┘
    ↓
PostgreSQL
```

Pooling improves efficiency and controls connection usage.

Too many PostgreSQL connections can cause resource pressure.

---

# 72. PgBouncer

PgBouncer is a lightweight PostgreSQL connection pooler.

```text
Applications
    │
    ▼
PgBouncer
    │
    ▼
PostgreSQL
```

It can reduce the number of backend PostgreSQL connections required by many clients.

---

# 73. Roles and Permissions

PostgreSQL uses roles for authentication and authorization.

Important privileges include:

```text
SELECT
INSERT
UPDATE
DELETE
REFERENCES
EXECUTE
USAGE
```

Example:

```sql
GRANT SELECT
ON users
TO reporting_user;
```

Follow least privilege.

---

# 74. Row-Level Security

Row-Level Security (RLS) can restrict which rows a user can access.

Useful in multi-tenant SaaS.

```text
Database
 ├── Tenant A rows
 ├── Tenant B rows
 └── Tenant C rows

Tenant A user
      ↓
   RLS Policy
      ↓
Only Tenant A rows
```

RLS is an additional database-level protection and should be designed together with application authorization.

---

# 75. Multi-Tenant Architecture

Common strategies:

### Database per tenant

Strong isolation but higher operational complexity.

### Schema per tenant

```text
database
 ├── tenant_a
 ├── tenant_b
 └── tenant_c
```

### Shared tables + tenant_id

```text
customers
 ├── tenant_id
 ├── customer_id
 └── ...
```

Shared tables are common for SaaS, often combined with RLS.

---

# 76. Normalization

Normalization reduces duplication and improves integrity.

Typical levels:

```text
1NF
2NF
3NF
BCNF
```

Example bad design:

```text
orders
 ├── order_id
 ├── customer_name
 ├── customer_email
 ├── product_name
 └── product_price
```

Better:

```text
customers
products
orders
order_items
```

---

# 77. Denormalization

Denormalization intentionally stores derived or repeated data to improve some read workloads.

Example:

```text
customer_summary
 ├── customer_id
 ├── total_orders
 └── total_spent
```

Trade-off:

```text
Normalization
 → less duplication
 → simpler integrity

Denormalization
 → potentially faster reads
 → more consistency/maintenance complexity
```

---

# 78. Partitioning

Partitioning divides one logical table into multiple physical partitions.

```text
events
 ├── events_2026_01
 ├── events_2026_02
 ├── events_2026_03
 └── ...
```

Common partitioning strategies:

```text
RANGE
LIST
HASH
```

Time-based range partitioning is common for event/log tables.

---

# 79. Partition Pruning

If a table is partitioned by date:

```sql
SELECT *
FROM events
WHERE created_at >= '2026-09-01'
  AND created_at < '2026-10-01';
```

PostgreSQL may access only relevant partitions.

```text
Query
 ↓
Partition Pruning
 ↓
September Partition
 ↓
Rows
```

---

# 80. WAL

WAL means **Write-Ahead Logging**.

Simplified:

```text
Transaction
    ↓
WAL Record
    ↓
Durable Commit
    ↓
Data Pages
```

WAL is central to:

- Crash recovery
- Durability
- Replication
- Point-in-time recovery

---

# 81. Crash Recovery

After a crash:

```text
PostgreSQL Restart
      ↓
Read WAL
      ↓
Replay required changes
      ↓
Recover consistent state
```

---

# 82. Backup

Backups protect against:

- Accidental deletion
- Corruption
- Application bugs
- Hardware failure
- Disaster

Common tools/approaches:

```text
pg_dump
pg_restore
pg_basebackup
WAL archiving
```

Logical backup:

```bash
pg_dump my_database > backup.sql
```

Restore:

```bash
psql my_database < backup.sql
```

Custom format:

```bash
pg_dump -Fc my_database > backup.dump
pg_restore -d my_database backup.dump
```

---

# 83. Replication

A simplified architecture:

```text
Primary
   │
   │ WAL / Replication
   ▼
Standby
   │
   ▼
Replicated Data
```

Uses:

- High availability
- Read scaling
- Disaster recovery

Replication is only one component of a complete HA strategy.

---

# 84. Streaming Replication

A primary can stream WAL changes to a standby.

```text
Primary
  │
  └── WAL stream ───► Standby
```

The standby can serve certain read workloads depending on configuration.

---

# 85. Failover

If the primary fails:

```text
Primary
   X
   │
   ▼
Standby
   │
   ▼
Promote
```

Automatic failover requires carefully designed operational tooling and procedures.

---

# 86. Point-in-Time Recovery

PITR combines a base backup with WAL archives.

```text
Base Backup
    ↓
WAL 1
    ↓
WAL 2
    ↓
WAL 3
    ↓
Desired Recovery Time
```

It can help recover from accidental changes such as deleting important records.

---

# 87. Database Migrations

Application schemas evolve.

Example:

```text
Migration 001
CREATE users

Migration 002
ADD email

Migration 003
CREATE orders

Migration 004
ADD indexes
```

Python applications commonly use Alembic with SQLAlchemy.

Good migrations should be:

- Versioned
- Reproducible
- Tested
- Reviewed
- Safe for production
- Designed for large-table behavior

---

# 88. Locking

PostgreSQL uses locks for concurrency control.

Example:

```sql
SELECT *
FROM accounts
WHERE account_id = 1
FOR UPDATE;
```

This can lock the selected row for update within the transaction.

Useful for inventory, account balances, and other concurrency-sensitive operations.

---

# 89. Deadlocks

A deadlock can occur when:

```text
Transaction A
  locks Row 1
  waits for Row 2

Transaction B
  locks Row 2
  waits for Row 1
```

PostgreSQL detects deadlocks and aborts one transaction.

Applications should be prepared to retry appropriate transactions.

A common prevention strategy is to acquire locks in a consistent order.

---

# 90. Advisory Locks

Advisory locks provide application-defined coordination.

Potential uses:

- Distributed jobs
- Preventing duplicate work
- Application-level synchronization

They should be designed carefully because PostgreSQL does not know the business meaning of an advisory lock.

---

# 91. SQL Injection

Never construct SQL by concatenating untrusted input.

Bad:

```python
query = "SELECT * FROM users WHERE email = '" + email + "'"
```

Use parameterized queries.

Conceptually:

```sql
SELECT *
FROM users
WHERE email = $1;
```

Parameters are supplied separately by the driver.

---

# 92. Security Architecture

```text
Application
    ↓
Authentication
    ↓
Authorization
    ↓
Parameterized Queries
    ↓
Database Role
    ↓
PostgreSQL
    ↓
RLS / Constraints / Privileges
```

Security should exist at multiple layers.

---

# 93. Real-World Case Study: E-Commerce

Tables:

```text
customers
products
orders
order_items
payments
shipments
```

Relationship:

```text
Customer
   ↓
Orders
   ↓
Order Items
   ↓
Products

Order
 ├── Payment
 └── Shipment
```

A purchase might use:

```sql
BEGIN;

INSERT INTO orders (...);

INSERT INTO order_items (...);

UPDATE products
SET stock = stock - 1
WHERE product_id = 101
  AND stock > 0;

COMMIT;
```

If a critical step fails:

```sql
ROLLBACK;
```

The transaction prevents a partially completed workflow when designed correctly.

---

# 94. Real-World Case Study: Banking

Tables:

```text
customers
accounts
transactions
```

Transfer:

```sql
BEGIN;

UPDATE accounts
SET balance = balance - 5000
WHERE account_id = 1;

UPDATE accounts
SET balance = balance + 5000
WHERE account_id = 2;

INSERT INTO transactions (...);

COMMIT;
```

Important concerns:

- ACID
- Concurrency
- Locking
- Auditability
- Authorization
- Recovery
- Idempotency

---

# 95. Real-World Case Study: SaaS

Possible schema:

```text
organizations
users
memberships
subscriptions
invoices
audit_logs
```

Architecture:

```text
Organization
 ├── Users
 ├── Subscription
 ├── Invoices
 └── Audit Logs
```

For shared multi-tenant tables:

```text
tenant_id
```

is commonly indexed and can be combined with RLS.

---

# 96. Real-World Case Study: AI RAG

A RAG system can use:

```text
users
conversations
messages
documents
document_chunks
audit_logs
```

with `document_chunks` containing:

```text
content
metadata JSONB
embedding VECTOR(...)
```

Architecture:

```text
User
 ↓
FastAPI
 ↓
LangGraph
 ├── LLM
 ├── Retriever
 └── Tools
       ↓
PostgreSQL
 ├── Relational Data
 ├── JSONB
 └── pgvector
```

This allows a single database to serve as a durable application system of record plus vector store for many moderate-scale AI applications.

---

# 97. Real-World Case Study: Analytics

Suppose an application produces millions of events.

```text
events
 ├── event_id
 ├── user_id
 ├── event_type
 ├── created_at
 └── metadata JSONB
```

Possible design:

```text
Partition by date
        +
Indexes based on queries
        +
JSONB for flexible metadata
        +
Materialized views for reports
```

---

# 98. N+1 Query Problem

Bad pattern:

```text
Query users
    ↓
For every user:
    Query orders
```

100 users can become:

```text
1 + 100 = 101 queries
```

Better approaches:

- Joins
- Batch queries
- Eager loading
- Aggregation
- Carefully designed repository methods

---

# 99. Pagination at Scale

Offset pagination:

```sql
SELECT *
FROM products
ORDER BY product_id
LIMIT 20
OFFSET 10000;
```

can become inefficient for very large offsets.

Keyset pagination:

```sql
SELECT *
FROM products
WHERE product_id > 10000
ORDER BY product_id
LIMIT 20;
```

This can be more efficient because PostgreSQL can continue from a known position.

---

# 100. Query Optimization Workflow

When a query is slow:

```text
1. Identify slow query
       ↓
2. Measure baseline
       ↓
3. EXPLAIN
       ↓
4. EXPLAIN ANALYZE
       ↓
5. Inspect scans / joins / sorting
       ↓
6. Check indexes
       ↓
7. Check statistics
       ↓
8. Rewrite if needed
       ↓
9. Benchmark again
```

Do not optimize purely by intuition.

---

# 101. Index Design Principles

Good practice:

- Index frequently filtered columns
- Index relevant join keys
- Use composite indexes for real query patterns
- Use partial indexes for selective subsets
- Use expression indexes when needed
- Monitor index usage

Avoid:

- Indexing every column
- Creating redundant indexes
- Ignoring write overhead
- Creating indexes without measuring workload

---

# 102. PostgreSQL vs Redis

PostgreSQL:

```text
Durable system of record
Transactions
Relational queries
Constraints
Complex SQL
```

Redis:

```text
Cache
Transient state
Rate limiting
Pub/Sub
Some queue workloads
```

They are often complementary.

```text
Application
 ├── Redis → fast transient data
 └── PostgreSQL → durable system of record
```

---

# 103. PostgreSQL vs SQLite

SQLite is:

- Embedded
- Serverless
- File-based
- Excellent for local/small applications

PostgreSQL is:

- Client-server
- Multi-user
- Feature-rich
- Strong for production backend workloads

```text
SQLite
Application
    ↓
Database File

PostgreSQL
Application
    ↓
PostgreSQL Server
    ↓
Database Storage
```

---

# 104. PostgreSQL vs MySQL

Both are mature relational databases.

PostgreSQL is especially known for:

- Advanced SQL
- Extensibility
- JSONB
- Rich data types
- Strong indexing
- Advanced transactional capabilities
- Powerful extensions

The best database depends on workload, team experience, operational requirements, and ecosystem.

---

# 105. Practical E-Commerce Schema

```sql
CREATE TABLE customers (
    customer_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    created_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE products (
    product_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    price NUMERIC(12,2) NOT NULL CHECK (price >= 0),
    stock INTEGER NOT NULL CHECK (stock >= 0)
);

CREATE TABLE orders (
    order_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id BIGINT NOT NULL
        REFERENCES customers(customer_id),
    created_at TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE order_items (
    order_id BIGINT REFERENCES orders(order_id),
    product_id BIGINT REFERENCES products(product_id),
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price NUMERIC(12,2) NOT NULL CHECK (unit_price >= 0),
    PRIMARY KEY (order_id, product_id)
);
```

ER diagram:

```text
customers
    │
    │ 1
    ▼
orders
    │
    │ 1
    ▼
order_items
    ▲
    │ N
    │
products
```

---

# 106. Real-World SQL Queries

### Customer order count

```sql
SELECT
    c.customer_id,
    c.name,
    COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY order_count DESC;
```

### Total customer spend

```sql
SELECT
    c.customer_id,
    c.name,
    SUM(oi.quantity * oi.unit_price) AS total_spend
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name
ORDER BY total_spend DESC;
```

### Top products

```sql
SELECT
    p.product_id,
    p.name,
    SUM(oi.quantity) AS units_sold
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.name
ORDER BY units_sold DESC
LIMIT 10;
```

### Monthly revenue

```sql
SELECT
    DATE_TRUNC('month', o.created_at) AS month,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY 1
ORDER BY 1;
```

### Customers without orders

```sql
SELECT
    c.customer_id,
    c.name
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
```

---

# 107. PostgreSQL + AI Application Architecture

A practical modern stack can look like:

```text
┌────────────────────────────┐
│          Client            │
└─────────────┬──────────────┘
              ↓
┌────────────────────────────┐
│          FastAPI           │
└─────────────┬──────────────┘
              ↓
┌────────────────────────────┐
│         LangGraph          │
│                            │
│ LLM ── Tools ── Retriever  │
└─────────────┬──────────────┘
              ↓
┌────────────────────────────┐
│        SQLAlchemy          │
└─────────────┬──────────────┘
              ↓
┌────────────────────────────┐
│        PostgreSQL          │
│                            │
│ Users                      │
│ Conversations              │
│ Documents                  │
│ Chunks                     │
│ JSONB                      │
│ Embeddings / pgvector      │
│ Audit Logs                 │
└────────────────────────────┘
```

This is especially useful for systems where application data and retrieval data need strong consistency and a shared operational database.

---

# 108. PostgreSQL CLI Commands

Using `psql`:

```text
\l          List databases
\c db       Connect to database
\dt         List tables
\d table    Describe table
\dn         List schemas
\du         List roles
\q          Quit
```

These are `psql` meta-commands, not standard SQL.

---

# 109. Useful SQL Commands

Create database:

```sql
CREATE DATABASE app;
```

Create table:

```sql
CREATE TABLE users (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL
);
```

Alter:

```sql
ALTER TABLE users
ADD COLUMN email TEXT;
```

Remove all rows:

```sql
TRUNCATE TABLE users;
```

Remove table:

```sql
DROP TABLE users;
```

---

# 110. DELETE vs TRUNCATE vs DROP

```text
DELETE
 → removes rows

TRUNCATE
 → removes table data efficiently

DROP
 → removes the database object
```

Examples:

```sql
DELETE FROM users WHERE id = 1;

TRUNCATE TABLE users;

DROP TABLE users;
```

Use destructive commands carefully.

---

# 111. Production PostgreSQL Checklist

## Schema

```text
[ ] Primary keys
[ ] Foreign keys
[ ] Appropriate data types
[ ] Constraints
[ ] Naming conventions
```

## Queries

```text
[ ] Parameterized queries
[ ] No accidental N+1
[ ] Efficient pagination
[ ] Appropriate joins
[ ] Avoid unnecessary SELECT *
```

## Indexes

```text
[ ] Query-driven indexes
[ ] Join-key indexes where appropriate
[ ] Composite indexes based on real patterns
[ ] No redundant indexes
```

## Transactions

```text
[ ] Correct boundaries
[ ] Appropriate isolation
[ ] Deadlock handling
[ ] Avoid long transactions
```

## Security

```text
[ ] Least privilege
[ ] Secure secrets
[ ] TLS where appropriate
[ ] Input validation
[ ] SQL injection protection
[ ] RLS where appropriate
```

## Reliability

```text
[ ] Backups
[ ] Recovery testing
[ ] WAL strategy
[ ] Replication where needed
[ ] Monitoring
```

---

# 112. PostgreSQL Learning Roadmap

```text
Level 1
SQL Basics
Tables
CRUD
Filtering
       ↓
Level 2
Joins
Aggregation
Subqueries
CTEs
Window Functions
       ↓
Level 3
Constraints
Relationships
Transactions
ACID
MVCC
       ↓
Level 4
Indexes
EXPLAIN
Optimization
Partitioning
       ↓
Level 5
JSONB
Arrays
Full-Text Search
Extensions
       ↓
Level 6
Functions
Procedures
Triggers
RLS
Security
       ↓
Level 7
WAL
Backup
Replication
Recovery
Monitoring
       ↓
Level 8
FastAPI
SQLAlchemy
Redis
pgvector
RAG
LangGraph
Agentic AI
```

---

# 113. Interview Questions

## Beginner

1. What is PostgreSQL?
2. PostgreSQL vs SQL?
3. What is a primary key?
4. What is a foreign key?
5. What is a schema?
6. What is a constraint?
7. What is `NOT NULL`?
8. What is `UNIQUE`?
9. What is `CHECK`?
10. What is `DEFAULT`?

## Intermediate

1. What is a transaction?
2. Explain ACID.
3. What is MVCC?
4. What is an index?
5. What is a composite index?
6. What is a partial index?
7. What is GIN?
8. What is `EXPLAIN ANALYZE`?
9. What is a CTE?
10. What are window functions?
11. What is UPSERT?
12. What is `RETURNING`?
13. What is a materialized view?
14. What is a trigger?
15. What is autovacuum?

## Advanced

1. Explain MVCC in PostgreSQL.
2. Explain WAL.
3. Explain isolation levels.
4. Explain query planning.
5. Explain index selection.
6. Explain partitioning.
7. Explain replication.
8. Explain PITR.
9. Explain RLS.
10. Explain connection pooling.
11. Explain deadlocks.
12. Explain advisory locks.
13. Explain JSONB indexing.
14. Explain pgvector.
15. Explain PostgreSQL architecture.

---

# 114. Interview Answer — Why PostgreSQL for AI?

PostgreSQL is useful for AI applications because it can combine relational data with capabilities such as JSONB, full-text search, and vector search through pgvector.

An AI application can store users, conversations, documents, chunks, metadata, permissions, audit logs, and embeddings in one operational database. This can simplify architecture for many moderate-scale systems.

However, database choice should still be based on workload, scale, latency, operational requirements, and search needs.

---

# 115. Interview Answer — What Is MVCC?

MVCC, or Multi-Version Concurrency Control, is PostgreSQL's approach to handling concurrent transactions using multiple row versions and visibility rules.

It allows many reads and writes to proceed with less blocking than a simplistic locking-only design.

---

# 116. Interview Answer — What Is WAL?

WAL stands for Write-Ahead Logging.

PostgreSQL records changes in WAL before corresponding data pages are considered durably persisted. WAL supports crash recovery, durability, replication, and point-in-time recovery.

---

# 117. Interview Answer — Are Indexes Always Good?

No.

Indexes improve some read workloads but consume storage and add write and maintenance overhead. An index should be created because a real workload benefits from it, and the effect should be validated using execution plans and measurements.

---

# 118. Interview Answer — What Is JSONB?

JSONB is PostgreSQL's binary JSON representation optimized for processing and indexing.

It is useful when an application has semi-structured data that does not fit naturally into fixed relational columns.

---

# 119. Interview Answer — How Would You Debug a Slow Query?

I would first identify and reproduce the slow query, inspect it using `EXPLAIN` and `EXPLAIN ANALYZE`, examine scans, joins, sorting, row estimates, indexes, statistics, and locking, then make a targeted change and benchmark again.

---

# 120. Final Mental Model

Think of PostgreSQL in layers:

```text
┌───────────────────────────────────────┐
│ DATA MODEL                            │
│ Tables • Keys • Relationships         │
├───────────────────────────────────────┤
│ QUERY                                 │
│ SQL • Joins • CTEs • Windows          │
├───────────────────────────────────────┤
│ RELIABILITY                           │
│ ACID • MVCC • WAL • Transactions      │
├───────────────────────────────────────┤
│ PERFORMANCE                           │
│ Indexes • Planner • Partitioning      │
├───────────────────────────────────────┤
│ PRODUCTION                            │
│ Security • Backup • Replication       │
│ Monitoring • Recovery                 │
├───────────────────────────────────────┤
│ MODERN APPLICATIONS                   │
│ JSONB • Search • pgvector • PostGIS   │
└───────────────────────────────────────┘
```

For Agentic AI:

```text
LLM
 ↓
LangGraph
 ↓
Tools / Retrieval
 ↓
SQLAlchemy
 ↓
PostgreSQL
 ├── Application Data
 ├── Conversations
 ├── Documents
 ├── JSONB Metadata
 ├── Audit Logs
 └── pgvector
```

> **The real skill in PostgreSQL is not memorizing SQL commands. It is understanding data modeling, integrity, transactions, concurrency, indexing, query planning, security, recovery, and how PostgreSQL fits into the complete application architecture.**

---

# 121. Final Cheat Sheet

```text
DATABASE
    CREATE DATABASE app;

TABLE
    CREATE TABLE users (...);

CRUD
    INSERT
    SELECT
    UPDATE
    DELETE

RELATIONSHIPS
    PRIMARY KEY
    FOREIGN KEY
    1:1
    1:N
    M:N

QUERYING
    WHERE
    JOIN
    GROUP BY
    HAVING
    CTE
    WINDOW FUNCTIONS

RELIABILITY
    TRANSACTION
    ACID
    MVCC
    WAL
    VACUUM

PERFORMANCE
    INDEX
    EXPLAIN
    EXPLAIN ANALYZE
    PARTITIONING

ADVANCED
    JSONB
    ARRAY
    FULL-TEXT SEARCH
    GIN
    GiST
    BRIN
    PostGIS
    pgvector

PRODUCTION
    ROLES
    PRIVILEGES
    RLS
    CONNECTION POOLING
    BACKUP
    RESTORE
    REPLICATION
    PITR
    MONITORING

AI
    PostgreSQL + pgvector
    PostgreSQL + RAG
    PostgreSQL + LangGraph
    PostgreSQL + FastAPI
    PostgreSQL + SQLAlchemy
```

> **PostgreSQL can act as the durable system of record for traditional backend systems and, with extensions such as pgvector, as a powerful data layer for modern RAG and Agentic AI applications.**
